# TTF Options — Dashboard Jupyter

Dashboard interactif pour le pricing des options TTF (gaz naturel europeen), base sur `black76_ttf.py`.

## Section 1 - Pricer

Reglez les parametres ci-dessous. Les prix Call / Put et les Greeks se recalculent automatiquement.

**Conventions**
- Forward, Strike : EUR/MWh
- Maturite : annees (ACT/365)
- Taux : decimal annualise (`0.03` = 3 %)
- Vol Black-76 : lognormale decimale (`0.50` = 50 %)
- Vol Bachelier : normale en EUR/MWh

In [ ]:
import ipywidgets as widgets
from IPython.display import display

import black76_ttf as b76m

In [ ]:
# ---- Widgets ----
style = {"description_width": "140px"}
layout = widgets.Layout(width="480px")

model = widgets.Dropdown(
    options=["Black-76", "Bachelier"], value="Black-76",
    description="Modele :", style=style, layout=layout,
)
forward = widgets.FloatSlider(
    value=35.0, min=1.0, max=150.0, step=0.5,
    description="Forward (EUR/MWh) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
strike = widgets.FloatSlider(
    value=35.0, min=1.0, max=150.0, step=0.5,
    description="Strike (EUR/MWh) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
maturity = widgets.FloatSlider(
    value=0.25, min=0.01, max=3.0, step=0.01,
    description="Maturite (annees) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
rate = widgets.FloatSlider(
    value=0.03, min=-0.02, max=0.10, step=0.0025,
    description="Taux (decimal) :", style=style, layout=layout,
    continuous_update=False, readout_format=".4f",
)
vol_b76 = widgets.FloatSlider(
    value=0.50, min=0.01, max=2.00, step=0.01,
    description="Vol (lognormale) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
vol_bach = widgets.FloatSlider(
    value=10.0, min=0.1, max=50.0, step=0.1,
    description="Vol (EUR/MWh) :", style=style, layout=layout,
    continuous_update=False, readout_format=".2f",
)
vol_bach.layout.display = "none"  # hidden until Bachelier is selected

output = widgets.Output()


def _fmt_greeks(g) -> str:
    return (
        f"  Delta : {g.delta:+.4f}\n"
        f"  Gamma : {g.gamma:+.6f}\n"
        f"  Vega  : {g.vega:+.4f}\n"
        f"  Theta : {g.theta:+.4f} / jour\n"
        f"  Rho   : {g.rho:+.4f} / pp\n"
        f"  Vanna : {g.vanna:+.6f}\n"
        f"  Volga : {g.volga:+.6f}"
    )


def _recompute(*_):
    if model.value == "Black-76":
        vol_b76.layout.display = ""
        vol_bach.layout.display = "none"
    else:
        vol_b76.layout.display = "none"
        vol_bach.layout.display = ""

    F = forward.value
    K = strike.value
    T = maturity.value
    r = rate.value

    with output:
        output.clear_output()
        try:
            if model.value == "Black-76":
                sigma = vol_b76.value
                call = b76m.b76_call(F, K, T, r, sigma)
                put = b76m.b76_put(F, K, T, r, sigma)
                g_call = b76m.b76_greeks(F, K, T, r, sigma, "call")
                g_put = b76m.b76_greeks(F, K, T, r, sigma, "put")
                vol_label = f"sigma = {sigma:.2%} (lognormale)"
            else:
                sigma_n = vol_bach.value
                call = b76m.bach_call(F, K, T, r, sigma_n)
                put = b76m.bach_put(F, K, T, r, sigma_n)
                g_call = b76m.bach_greeks(F, K, T, r, sigma_n, "call")
                g_put = b76m.bach_greeks(F, K, T, r, sigma_n, "put")
                vol_label = f"sigma_n = {sigma_n:.2f} EUR/MWh (normale)"

            print(f"=== {model.value} - {vol_label} ===")
            print(f"F = {F:.2f}   K = {K:.2f}   T = {T:.4f} y   r = {r:.4f}")
            print("")
            print(f"Prix Call : {call:.4f} EUR/MWh")
            print(f"Prix Put  : {put:.4f} EUR/MWh")
            print("")
            print("Greeks Call :")
            print(_fmt_greeks(g_call))
            print("")
            print("Greeks Put :")
            print(_fmt_greeks(g_put))
        except Exception as exc:
            print(f"Erreur de calcul : {exc}")


for w in (model, forward, strike, maturity, rate, vol_b76, vol_bach):
    w.observe(_recompute, names="value")

_recompute()

display(widgets.VBox([
    model, forward, strike, maturity, rate, vol_b76, vol_bach,
    widgets.HTML("<hr>"),
    output,
]))

## Section 2 - Structures

Selectionnez une des 10 structures multi-leg (Black-76). Les strikes s'adaptent dynamiquement.

**Structures disponibles**
1. Straddle (1 strike)
2. Strangle (K_put, K_call)
3. Bull Call Spread (K_lo, K_hi)
4. Bear Put Spread (K_lo, K_hi)
5. Butterfly (K_lo, K_mid, K_hi)
6. Condor (K1, K2, K3, K4)
7. Collar (K_put, K_call)
8. Risk Reversal (K_put, K_call)
9. Calendar Spread (K ; T_far = T + 3 mois)
10. Ratio Spread 1x2 (K_lo, K_hi)

Taux fixe a r = 3 % ; le graphique montre le P&L a expiration en EUR/MWh.

In [ ]:
import matplotlib.pyplot as plt

import structures_ttf as st

In [ ]:
# ---- Structure -> strike labels ----
STRUCT_CONFIG = {
    "Straddle":         ["K"],
    "Strangle":         ["K_put", "K_call"],
    "Bull Call Spread": ["K_lo", "K_hi"],
    "Bear Put Spread":  ["K_lo", "K_hi"],
    "Butterfly":        ["K_lo", "K_mid", "K_hi"],
    "Condor":           ["K1", "K2", "K3", "K4"],
    "Collar":           ["K_put", "K_call"],
    "Risk Reversal":    ["K_put", "K_call"],
    "Calendar Spread":  ["K"],
    "Ratio Spread 1x2": ["K_lo", "K_hi"],
}

R_FIXED = 0.03  # risk-free rate used throughout Section 2


def _default_strikes(name: str, F: float) -> list[float]:
    """Reasonable default strikes around the forward for a given structure."""
    if name == "Straddle":          return [F]
    if name == "Strangle":          return [F - 3, F + 3]
    if name == "Bull Call Spread":  return [F - 1, F + 3]
    if name == "Bear Put Spread":   return [F - 3, F + 1]
    if name == "Butterfly":         return [F - 4, F, F + 4]
    if name == "Condor":            return [F - 5, F - 2, F + 2, F + 5]
    if name == "Collar":            return [F - 3, F + 3]
    if name == "Risk Reversal":     return [F - 3, F + 3]
    if name == "Calendar Spread":   return [F]
    if name == "Ratio Spread 1x2": return [F, F + 3]
    return [F]


def _run_structure(name: str, F: float, ks: list[float], T: float, sigma: float):
    """Dispatch to the correct structures_ttf function."""
    r = R_FIXED
    if name == "Straddle":
        return st.straddle(F, K=ks[0], T=T, r=r, sigma=sigma)
    if name == "Strangle":
        return st.strangle(F, K_put=ks[0], K_call=ks[1], T=T, r=r, sigma=sigma)
    if name == "Bull Call Spread":
        return st.bull_call_spread(F, K_lo=ks[0], K_hi=ks[1], T=T, r=r, sigma=sigma)
    if name == "Bear Put Spread":
        return st.bear_put_spread(F, K_lo=ks[0], K_hi=ks[1], T=T, r=r, sigma=sigma)
    if name == "Butterfly":
        return st.butterfly(F, K_lo=ks[0], K_mid=ks[1], K_hi=ks[2], T=T, r=r, sigma=sigma)
    if name == "Condor":
        return st.condor(F, K1=ks[0], K2=ks[1], K3=ks[2], K4=ks[3], T=T, r=r, sigma=sigma)
    if name == "Collar":
        return st.collar(F, K_put=ks[0], K_call=ks[1], T=T, r=r, sigma=sigma)
    if name == "Risk Reversal":
        return st.risk_reversal(F, K_put=ks[0], K_call=ks[1], T=T, r=r, sigma=sigma)
    if name == "Calendar Spread":
        T_near = T
        T_far = T + 0.25
        return st.calendar_spread(F, K=ks[0], T_far=T_far, T_near=T_near, r=r, sigma=sigma)
    if name == "Ratio Spread 1x2":
        return st.ratio_spread(F, K_lo=ks[0], K_hi=ks[1], T=T, r=r, sigma=sigma, ratio=2)
    raise ValueError(f"Unknown structure: {name}")


# ---- Widgets ----
st_style = {"description_width": "160px"}
st_layout = widgets.Layout(width="480px")

st_structure = widgets.Dropdown(
    options=list(STRUCT_CONFIG.keys()), value="Straddle",
    description="Structure :", style=st_style, layout=st_layout,
)
st_forward = widgets.FloatSlider(
    value=35.0, min=1.0, max=150.0, step=0.5,
    description="Forward (EUR/MWh) :", style=st_style, layout=st_layout,
    continuous_update=False, readout_format=".2f",
)
st_vol = widgets.FloatSlider(
    value=0.50, min=0.05, max=2.00, step=0.01,
    description="Vol (lognormale) :", style=st_style, layout=st_layout,
    continuous_update=False, readout_format=".2f",
)
st_maturity = widgets.FloatSlider(
    value=0.25, min=0.02, max=2.0, step=0.01,
    description="Maturite (annees) :", style=st_style, layout=st_layout,
    continuous_update=False, readout_format=".2f",
)
st_strikes = [
    widgets.FloatSlider(
        value=35.0, min=1.0, max=150.0, step=0.25,
        description=f"K{i+1} :", style=st_style, layout=st_layout,
        continuous_update=False, readout_format=".2f",
    )
    for i in range(4)
]

st_output = widgets.Output()

_updating = {"flag": False}


def _configure_strike_widgets(name: str, F: float, reset_values: bool) -> None:
    labels = STRUCT_CONFIG[name]
    defaults = _default_strikes(name, F)
    _updating["flag"] = True
    try:
        for i, w in enumerate(st_strikes):
            if i < len(labels):
                w.description = f"{labels[i]} :"
                w.layout.display = ""
                if reset_values:
                    w.value = float(defaults[i])
            else:
                w.layout.display = "none"
    finally:
        _updating["flag"] = False


def _plot_pnl(ax, result, F: float) -> None:
    xs = [pt[0] for pt in result.pnl_at_expiry]
    ys = [pt[1] for pt in result.pnl_at_expiry]
    ax.plot(xs, ys, color="tab:blue", linewidth=2.0, label="P&L a expiry")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.axvline(F, color="gray", linestyle="--", linewidth=0.8, label=f"Forward = {F:.2f}")
    for be in result.breakevens:
        ax.axvline(be, color="tab:red", linestyle=":", linewidth=0.8)
    for leg in result.legs:
        ax.axvline(leg.K, color="tab:green", alpha=0.25, linewidth=0.8)
    ax.fill_between(xs, ys, 0, where=[y >= 0 for y in ys], color="tab:green", alpha=0.15)
    ax.fill_between(xs, ys, 0, where=[y <  0 for y in ys], color="tab:red",   alpha=0.15)
    ax.set_xlabel("Forward a expiration F_T (EUR/MWh)")
    ax.set_ylabel("P&L (EUR/MWh)")
    ax.set_title(f"{result.name} - P&L a expiry")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=9)


def _structure_refresh(*_):
    if _updating["flag"]:
        return
    with st_output:
        st_output.clear_output(wait=True)
        name = st_structure.value
        F = st_forward.value
        T = st_maturity.value
        sigma = st_vol.value
        labels = STRUCT_CONFIG[name]
        ks = [st_strikes[i].value for i in range(len(labels))]
        try:
            result = _run_structure(name, F, ks, T, sigma)
        except Exception as exc:
            print(f"Erreur : {exc}")
            return
        premium_label = "debit" if result.price >= 0 else "credit"
        be = ", ".join(f"{b:.3f}" for b in result.breakevens) or "aucun"
        mp = f"{result.max_profit:.4f}" if result.max_profit != float('inf')  else "+inf"
        ml = f"{result.max_loss:.4f}"   if result.max_loss   != float('-inf') else "-inf"
        print(f"=== {result.name} ===")
        print(f"Net premium : {result.price:+.4f} EUR/MWh ({premium_label})")
        print(f"Greeks nets : delta={result.delta:+.4f}  gamma={result.gamma:+.6f}  "
              f"vega={result.vega:+.4f}  theta={result.theta:+.4f}/jour")
        print(f"Breakevens  : {be}")
        print(f"Max profit  : {mp}   Max loss : {ml}")
        fig, ax = plt.subplots(figsize=(9, 4.5))
        _plot_pnl(ax, result, F)
        plt.tight_layout()
        plt.show()


def _on_structure_change(change):
    _configure_strike_widgets(change["new"], st_forward.value, reset_values=True)
    _structure_refresh()


st_structure.observe(_on_structure_change, names="value")
for w in (st_forward, st_vol, st_maturity, *st_strikes):
    w.observe(_structure_refresh, names="value")

_configure_strike_widgets(st_structure.value, st_forward.value, reset_values=True)
_structure_refresh()

display(widgets.VBox([
    st_structure, st_forward, st_maturity, st_vol,
    *st_strikes,
    widgets.HTML("<hr>"),
    st_output,
]))

## Section 3 - Vol Surface

Surface de volatilite implicite TTF, parametree (ATM term-structure + smile log-moneyness).
Visualisation 3D matplotlib : axe X = strikes, axe Y = maturites, axe Z = vol implicite lognormale.

**Parametrage**
- ATM long terme `sigma_inf` et pic court terme `sigma_0 = sigma_inf + delta_sigma` avec decroissance exponentielle de taux `kappa` :
  `atm(T) = sigma_inf + delta_sigma * exp(-kappa * T)`
- Smile : `vol(K, T) = atm(T) + skew * m + wings * m^2`, ou `m = log(K/F) / sqrt(T)`
- Le forward `F` est suppose constant sur l'horizon (simplification ; une vraie courbe pourrait etre chargee via `ttf_market_data`).

In [ ]:
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3d projection)

In [ ]:
# ---- Parametric TTF vol surface ----
def vol_surface_grid(
    F: float,
    sigma_inf: float,
    delta_sigma: float,
    kappa: float,
    skew: float,
    wings: float,
    strike_pct: float,
    T_max: float,
    n_k: int = 35,
    n_t: int = 25,
):
    """Return (K_grid, T_grid, vol_grid) as 2D numpy meshes."""
    Ks = np.linspace(F * (1 - strike_pct), F * (1 + strike_pct), n_k)
    Ts = np.linspace(max(T_max / n_t, 1 / 52), T_max, n_t)  # avoid T=0
    K_mesh, T_mesh = np.meshgrid(Ks, Ts)
    atm = sigma_inf + delta_sigma * np.exp(-kappa * T_mesh)
    m = np.log(K_mesh / F) / np.sqrt(T_mesh)
    vol = atm + skew * m + wings * m**2
    vol = np.clip(vol, 0.01, 5.0)  # sanity floor/cap
    return K_mesh, T_mesh, vol


# ---- Widgets ----
vs_style = {"description_width": "180px"}
vs_layout = widgets.Layout(width="480px")

vs_forward = widgets.FloatSlider(
    value=35.0, min=5.0, max=100.0, step=0.5,
    description="Forward (EUR/MWh) :", style=vs_style, layout=vs_layout,
    continuous_update=False, readout_format=".2f",
)
vs_sigma_inf = widgets.FloatSlider(
    value=0.38, min=0.10, max=1.00, step=0.01,
    description="sigma_inf (ATM long) :", style=vs_style, layout=vs_layout,
    continuous_update=False, readout_format=".2f",
)
vs_delta_sigma = widgets.FloatSlider(
    value=0.30, min=0.00, max=1.00, step=0.01,
    description="delta_sigma (ATM court) :", style=vs_style, layout=vs_layout,
    continuous_update=False, readout_format=".2f",
)
vs_kappa = widgets.FloatSlider(
    value=2.0, min=0.1, max=8.0, step=0.1,
    description="kappa (decroissance) :", style=vs_style, layout=vs_layout,
    continuous_update=False, readout_format=".2f",
)
vs_skew = widgets.FloatSlider(
    value=-0.08, min=-0.40, max=0.40, step=0.01,
    description="skew :", style=vs_style, layout=vs_layout,
    continuous_update=False, readout_format=".2f",
)
vs_wings = widgets.FloatSlider(
    value=0.10, min=0.00, max=0.50, step=0.01,
    description="wings (convexite) :", style=vs_style, layout=vs_layout,
    continuous_update=False, readout_format=".2f",
)
vs_strike_pct = widgets.FloatSlider(
    value=0.50, min=0.10, max=0.90, step=0.05,
    description="range strikes (+/- %) :", style=vs_style, layout=vs_layout,
    continuous_update=False, readout_format=".0%",
)
vs_tmax = widgets.FloatSlider(
    value=2.0, min=0.25, max=5.0, step=0.25,
    description="T max (annees) :", style=vs_style, layout=vs_layout,
    continuous_update=False, readout_format=".2f",
)

vs_output = widgets.Output()


def _volsurface_refresh(*_):
    with vs_output:
        vs_output.clear_output(wait=True)
        F = vs_forward.value
        K_mesh, T_mesh, vol = vol_surface_grid(
            F=F,
            sigma_inf=vs_sigma_inf.value,
            delta_sigma=vs_delta_sigma.value,
            kappa=vs_kappa.value,
            skew=vs_skew.value,
            wings=vs_wings.value,
            strike_pct=vs_strike_pct.value,
            T_max=vs_tmax.value,
        )
        fig = plt.figure(figsize=(10, 6))
        ax = fig.add_subplot(111, projection="3d")
        surf = ax.plot_surface(
            K_mesh, T_mesh, vol,
            cmap="viridis", edgecolor="none", alpha=0.92,
            rstride=1, cstride=1, antialiased=True,
        )
        ax.set_xlabel("Strike K (EUR/MWh)")
        ax.set_ylabel("Maturite T (annees)")
        ax.set_zlabel("Vol implicite (lognormale)")
        ax.set_title(f"Vol surface TTF parametrique - F = {F:.2f} EUR/MWh")
        fig.colorbar(surf, shrink=0.6, aspect=12, pad=0.08, label="sigma")
        ax.view_init(elev=22, azim=-60)
        plt.tight_layout()
        plt.show()
        print(
            f"ATM(T=0)  = {vs_sigma_inf.value + vs_delta_sigma.value:.2%}    "
            f"ATM(T=inf) = {vs_sigma_inf.value:.2%}"
        )
        print(
            f"Range vol = [{vol.min():.2%}, {vol.max():.2%}]    "
            f"Grille : {vol.shape[1]} strikes x {vol.shape[0]} maturites"
        )


for w in (vs_forward, vs_sigma_inf, vs_delta_sigma, vs_kappa,
          vs_skew, vs_wings, vs_strike_pct, vs_tmax):
    w.observe(_volsurface_refresh, names="value")

_volsurface_refresh()

display(widgets.VBox([
    vs_forward, vs_sigma_inf, vs_delta_sigma, vs_kappa,
    vs_skew, vs_wings, vs_strike_pct, vs_tmax,
    widgets.HTML("<hr>"),
    vs_output,
]))